# TP4 — Detección de eventos atípicos en GDELT 2.0

**Curso:** Introducción a la Ingeniería de Datos — Doctorado
**Autor:** Marcos Kantor

En el TP1 procesamos el flujo de GDELT 2.0, en el TP2 diseñamos un almacén de datos para el negocio periodístico y en el TP3 analizamos el comportamiento multivariante del ecosistema global de noticias. En este TP buscamos **eventos "raros"** que valga la pena investigar.

**Pipeline del notebook**

| Etapa | Qué se hace | Producto |
|---|---|---|
| 1. Ingesta | Descarga de las tablas *Event* (`export.CSV.zip`) de 15 minutos de los últimos 7 días desde `masterfilelist.txt`, verificación MD5 y desnormalización con los libros de códigos CAMEO/FIPS | `gdelt_obt_7d.parquet` (One Big Table) |
| 2.1 Nulos y ceros | Diagnóstico de faltantes y ceros, criterio de descarte/imputación por variable | matriz analítica limpia |
| 2.2 Cardinalidad | Categorías con frecuencia < 5 % → `OTHER` | categóricas reducidas |
| 2.3 Variables indicadoras | *One-hot encoding* (dummies) con categoría de referencia | matriz numérica |
| 2.4 ACP | Escalado + Análisis por Componentes Principales, selección del número mínimo de componentes | matriz de *scores* |
| 2.5 Atípicos | T² de Hotelling + SPE (Q) + Isolation Forest, consenso por votos | `gdelt_atipicos.parquet` / `.csv` |

## Declaración de uso de Inteligencia Artificial

| Herramienta | Propósito |
|---|---|
| **Claude Opus 5.5** (Anthropic, vía Claude Code) | Generación del código fuente Python del notebook (ingesta, limpieza, ACP y detección de atípicos) y borrador de las celdas Markdown explicativas. |

Todas las decisiones metodológicas (umbral del 5 % de cardinalidad, criterio de imputación, regla de selección de componentes, nivel de significancia de los límites de control y regla de consenso) fueron definidas, revisadas y validadas por el autor. Los resultados numéricos se obtienen ejecutando el notebook; no fueron generados por la IA.

### Guía de instrucciones para reproducir el resultado (*Human-in-the-loop*)

La siguiente secuencia de instrucciones permite reproducir el notebook con la herramienta de IA. Cada paso indica la instrucción dada y la **verificación humana** realizada antes de pasar al siguiente.

1. **Contexto y arquitectura.**
   *"Actuá como ingeniero de datos. Necesito un notebook de Google Colab en Python que construya un pipeline batch sobre la tabla de hechos Event de GDELT 2.0 (61 campos, codebook V2.0). El pipeline tiene etapas: ingesta → One Big Table desnormalizada → limpieza → reducción de cardinalidad → variables indicadoras → ACP → detección de atípicos multivariantes. Usá pandas, pyarrow, scikit-learn y scipy; todo parametrizado en una celda de configuración."*
   ✔ Verifiqué que los 61 nombres de columna coincidan con el codebook GDELT-Event_Codebook-V2.0 y que los códigos CAMEO (`EventCode`, `EventRootCode`) se lean como texto para no perder ceros a la izquierda.

2. **Ingesta incremental e idempotente.**
   *"Leé `masterfilelist.txt` en streaming, filtrá solo los archivos `.export.CSV.zip` cuya marca temporal (YYYYMMDDHHMMSS) esté dentro de una ventana deslizante de 7 días en UTC, descargalos en paralelo con reintentos, validá el MD5 y el tamaño informados en la lista maestra, cacheá en disco y registrá los intervalos de 15 minutos faltantes."*
   ✔ Controlé la completitud: 7 × 96 = 672 intervalos esperados vs. descargados.

3. **Desnormalización (One Big Table).**
   *"Construí una OBT: a la tabla de hechos agregale las descripciones de las dimensiones (lookups CAMEO de eventos, países, tipos de actor, grupos, religiones, etnias; FIPS para geografía; QuadClass y Geo_Type), atributos derivados de tiempo (fecha de publicación, hora, día de semana, días de retraso entre SQLDATE y DATEADDED) y el dominio de la fuente. Eliminá duplicados por GLOBALEVENTID y persistí en Parquet con compresión snappy."*
   ✔ Revisé que el *grain* sea un evento por fila y que no se multipliquen filas al desnormalizar (uso de `map` en lugar de `merge`).

4. **Calidad de datos: nulos y ceros.**
   *"Generá un perfil de faltantes con % de nulos y % de ceros por columna. Distinguí ceros estructurales (GoldsteinScale y AvgTone = 0 son valores válidos, neutros) de ceros que representan faltantes (lat = long = 0, 'Null Island'). Para categóricas, tratá el nulo como categoría 'UNKNOWN' porque es faltante no aleatorio (MNAR); imputá coordenadas por mediana del país del ActionGeo; imputá GoldsteinScale por la moda del EventCode, ya que es función determinística del código CAMEO; descartá el resto y reportá cuántas filas se pierden."*
   ✔ Validé que el descarte de filas sea marginal (< 5 %) antes de aceptar el criterio.

5. **Transformaciones.**
   *"Aplicá log1p a las variables de conteo con asimetría fuerte (NumMentions, NumSources, NumArticles, DiasRetraso) y convertí lat/long a coordenadas cartesianas sobre la esfera unitaria para evitar la discontinuidad en ±180°. Mostrá la asimetría antes y después."*

6. **Reducción de cardinalidad y variables indicadoras.**
   *"Para cada categórica agrupá en 'OTHER' las categorías con frecuencia relativa menor al 5 % de la muestra; mostrá cardinalidad antes/después. Luego hacé one-hot encoding eliminando una categoría de referencia ('OTHER' si existe, si no la más frecuente) para evitar la colinealidad perfecta (trampa de las dummies)."*

7. **ACP.**
   *"Estandarizá (z-score) la matriz, ajustá PCA con todos los componentes, graficá scree plot y varianza acumulada, y compará criterios: varianza acumulada ≥ 80 % y ≥ 90 %, Kaiser (autovalor > 1) y codo. Elegí el número mínimo de componentes con el criterio de 80 % de varianza explicada y mostrá las cargas (loadings)."*
   ✔ Revisé que la cantidad elegida sea coherente entre criterios e interpreté las primeras componentes.

8. **Atípicos multivariantes.**
   *"En el subespacio de K componentes calculá la T² de Hotelling con límite χ²(0,999; K) y el error cuadrático de predicción SPE (estadístico Q) con la aproximación de Box; como método no paramétrico complementario usá Isolation Forest sobre los scores con el 0,1 % superior. Marcá como atípico de consenso al evento con ≥ 2 votos, rankealo por el promedio de percentiles, explicá cada atípico con las variables de mayor z-score y residuo, y exportalo a una tabla aparte en Parquet y CSV con todos los atributos de la OBT."*
   ✔ Inspeccioné manualmente los 25 atípicos más extremos abriendo su `SOURCEURL`.

## 0. Configuración del entorno

In [ ]:
# En Colab todas estas librerías ya están instaladas; se fuerza pyarrow por las dudas.
!pip -q install pyarrow

In [ ]:
import csv, hashlib, io, re, time, zipfile, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# ----------------------------- parámetros del pipeline -----------------------------
DIAS          = 7          # ventana deslizante de ingesta
MAX_ARCHIVOS  = None       # p.ej. 96 para una prueba rápida (1 día); None = ventana completa
MAX_WORKERS   = 16         # descargas en paralelo
UMBRAL_RARO   = 0.05       # categorías con frecuencia < 5 % -> 'OTHER'
UMBRAL_VAR    = 0.80       # varianza acumulada mínima para elegir K componentes
ALPHA         = 0.999      # nivel de confianza de los límites de control (T², SPE)
FRAC_IF       = 0.001      # fracción superior marcada por Isolation Forest
SEED          = 42

DATA_DIR = Path("/content/gdelt_tp4") if Path("/content").exists() else Path("gdelt_tp4")
RAW_DIR  = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OBT_PATH = DATA_DIR / "gdelt_obt_7d.parquet"
ATI_PATH = DATA_DIR / "gdelt_atipicos.parquet"
print("Directorio de trabajo:", DATA_DIR.resolve())

## 1. Ingesta de datos

### 1.1 Esquema de la tabla *Event* (GDELT 2.0)

Cada archivo `YYYYMMDDHHMMSS.export.CSV.zip` contiene un TSV sin encabezado con **61 campos** (codebook V2.0). Los códigos CAMEO y de país se leen como **texto** para conservar ceros a la izquierda (`"010"` ≠ `10`).

In [ ]:
ACTOR = ["Code", "Name", "CountryCode", "KnownGroupCode", "EthnicCode",
         "Religion1Code", "Religion2Code", "Type1Code", "Type2Code", "Type3Code"]
GEO   = ["Type", "FullName", "CountryCode", "ADM1Code", "ADM2Code", "Lat", "Long", "FeatureID"]

COLS = (["GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate"]
        + [f"Actor1{c}" for c in ACTOR] + [f"Actor2{c}" for c in ACTOR]
        + ["IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode", "QuadClass",
           "GoldsteinScale", "NumMentions", "NumSources", "NumArticles", "AvgTone"]
        + [f"Actor1Geo_{c}" for c in GEO] + [f"Actor2Geo_{c}" for c in GEO]
        + [f"ActionGeo_{c}" for c in GEO]
        + ["DATEADDED", "SOURCEURL"])
assert len(COLS) == 61

COLS_NUM = (["IsRootEvent", "QuadClass", "GoldsteinScale", "NumMentions", "NumSources",
             "NumArticles", "AvgTone", "Year", "FractionDate"]
            + [f"{p}Geo_{c}" for p in ("Actor1", "Actor2", "Action") for c in ("Type", "Lat", "Long")])

### 1.2 Selección de archivos desde `masterfilelist.txt`

La lista maestra tiene una línea por archivo: `tamaño md5 url`. Se lee en *streaming* (es grande) y se filtran sólo las tablas de evento (`export.CSV.zip`) dentro de la ventana de 7 días en UTC.

In [ ]:
MASTER_URL = "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt"

def listar_exports(dias: int) -> pd.DataFrame:
    corte = (datetime.now(timezone.utc) - timedelta(days=dias)).strftime("%Y%m%d%H%M%S")
    filas = []
    with requests.get(MASTER_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        for linea in r.iter_lines(decode_unicode=True):
            if not linea or not linea.endswith(".export.CSV.zip"):
                continue
            partes = linea.split()
            if len(partes) != 3:
                continue
            size, md5, url = partes
            ts = url.rsplit("/", 1)[-1].split(".")[0]
            if ts >= corte:
                filas.append((ts, int(size), md5, url))
    return (pd.DataFrame(filas, columns=["timestamp", "size", "md5", "url"])
              .drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True))

archivos = listar_exports(DIAS)
if MAX_ARCHIVOS:
    archivos = archivos.tail(MAX_ARCHIVOS).reset_index(drop=True)

esperados = DIAS * 96 if not MAX_ARCHIVOS else MAX_ARCHIVOS
print(f"Archivos en ventana: {len(archivos)} (esperados ≈ {esperados})")
print("Desde", archivos.timestamp.min(), "hasta", archivos.timestamp.max())
archivos.head()

### 1.3 Descarga paralela con verificación de integridad

Cada archivo se descarga con reintentos, se valida contra el **MD5** de la lista maestra y se cachea en disco (re-ejecutar el notebook no vuelve a descargar). El parseo convierte tipos por archivo para controlar el uso de memoria.

In [ ]:
def descargar(row) -> Path:
    destino = RAW_DIR / row.url.rsplit("/", 1)[-1]
    if destino.exists() and destino.stat().st_size == row.size:
        return destino
    err = None
    for intento in range(4):
        try:
            r = requests.get(row.url, timeout=90)
            r.raise_for_status()
            if hashlib.md5(r.content).hexdigest() != row.md5:
                raise ValueError("MD5 no coincide")
            destino.write_bytes(r.content)
            return destino
        except Exception as e:
            err = e
            time.sleep(2 * (intento + 1))
    raise RuntimeError(f"{row.url}: {err}")

def leer_export(path: Path, ts: str) -> pd.DataFrame:
    with zipfile.ZipFile(path) as z, z.open(z.namelist()[0]) as f:
        df = pd.read_csv(f, sep="\t", header=None, names=COLS, dtype=str,
                         quoting=csv.QUOTE_NONE, on_bad_lines="skip",
                         keep_default_na=False, na_values=[""],
                         encoding="utf-8", encoding_errors="replace")
    for c in COLS_NUM:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["GLOBALEVENTID"] = pd.to_numeric(df["GLOBALEVENTID"], errors="coerce")
    df["ArchivoOrigen"] = ts
    return df

def procesar(row):
    return leer_export(descargar(row), row.timestamp)

partes, fallidos = [], []
with ThreadPoolExecutor(MAX_WORKERS) as ex:
    futuros = {ex.submit(procesar, row): row.timestamp for row in archivos.itertuples()}
    for fut in tqdm(as_completed(futuros), total=len(futuros), desc="Event 15'"):
        try:
            partes.append(fut.result())
        except Exception as e:
            fallidos.append((futuros[fut], str(e)))

eventos = pd.concat(partes, ignore_index=True)
del partes
print(f"Intervalos OK: {len(archivos) - len(fallidos)} | fallidos: {len(fallidos)}")
print(f"Filas leídas: {len(eventos):,}")

### 1.4 Construcción de la One Big Table (OBT)

La OBT **desnormaliza** el modelo estrella del TP2: a la tabla de hechos se le agregan los atributos descriptivos de las dimensiones (libros de códigos CAMEO / FIPS publicados por GDELT) y atributos derivados de tiempo y fuente. Se usa `Series.map` con diccionarios para garantizar que el *grain* (un evento por fila) no cambie.

In [ ]:
LOOKUP_URL = "https://www.gdeltproject.org/data/lookups/{}"

def cargar_lookup(nombre: str) -> dict:
    try:
        lk = pd.read_csv(LOOKUP_URL.format(nombre), sep="\t", header=None, usecols=[0, 1],
                         names=["code", "label"], dtype=str, quoting=csv.QUOTE_NONE,
                         encoding_errors="replace")
        lk = lk.dropna()
        lk = lk[~lk["code"].str.upper().str.contains("CODE")]          # descarta encabezado
        return dict(zip(lk["code"].str.strip(), lk["label"].str.strip()))
    except Exception as e:
        print(f"[aviso] no se pudo cargar {nombre}: {e}")
        return {}

LK = {n: cargar_lookup(f"{n}.txt") for n in
      ["CAMEO.eventcodes", "CAMEO.country", "CAMEO.type", "CAMEO.knowngroup",
       "CAMEO.ethnic", "CAMEO.religion", "FIPS.country"]}
print({k: len(v) for k, v in LK.items()})

QUAD = {1: "Verbal Cooperation", 2: "Material Cooperation", 3: "Verbal Conflict", 4: "Material Conflict"}
GEOT = {1: "COUNTRY", 2: "USSTATE", 3: "USCITY", 4: "WORLDCITY", 5: "WORLDSTATE"}

In [ ]:
obt = eventos.drop_duplicates("GLOBALEVENTID").copy()
del eventos
print("Filas tras eliminar duplicados por GLOBALEVENTID:", f"{len(obt):,}")

# --- dimensión Evento (CAMEO)
for c in ["EventCode", "EventBaseCode", "EventRootCode"]:
    obt[c.replace("Code", "Desc")] = obt[c].map(LK["CAMEO.eventcodes"])
obt["QuadClassDesc"] = obt["QuadClass"].map(QUAD)

# --- dimensión Actor
for a in ["Actor1", "Actor2"]:
    obt[f"{a}CountryDesc"]   = obt[f"{a}CountryCode"].map(LK["CAMEO.country"])
    obt[f"{a}Type1Desc"]     = obt[f"{a}Type1Code"].map(LK["CAMEO.type"])
    obt[f"{a}KnownGroupDesc"] = obt[f"{a}KnownGroupCode"].map(LK["CAMEO.knowngroup"])
    obt[f"{a}EthnicDesc"]    = obt[f"{a}EthnicCode"].map(LK["CAMEO.ethnic"])
    obt[f"{a}Religion1Desc"] = obt[f"{a}Religion1Code"].map(LK["CAMEO.religion"])

# --- dimensión Geografía (FIPS)
for g in ["Actor1Geo", "Actor2Geo", "ActionGeo"]:
    obt[f"{g}_CountryDesc"] = obt[f"{g}_CountryCode"].map(LK["FIPS.country"])
    obt[f"{g}_TypeDesc"]    = obt[f"{g}_Type"].map(GEOT)

# --- dimensión Tiempo
obt["FechaEvento"]      = pd.to_datetime(obt["SQLDATE"], format="%Y%m%d", errors="coerce")
obt["FechaPublicacion"] = pd.to_datetime(obt["DATEADDED"], format="%Y%m%d%H%M%S", errors="coerce")
obt["DiasRetraso"]      = (obt["FechaPublicacion"].dt.normalize() - obt["FechaEvento"]).dt.days
obt["HoraPublicacion"]  = obt["FechaPublicacion"].dt.hour
obt["DiaSemana"]        = obt["FechaPublicacion"].dt.day_name()

# --- dimensión Fuente
obt["Dominio"] = obt["SOURCEURL"].str.extract(r"https?://(?:www\.)?([^/:]+)", expand=False).str.lower()

print(obt.shape)
obt.head(3)

In [ ]:
obt.to_parquet(OBT_PATH, engine="pyarrow", compression="snappy", index=False)
print(f"OBT guardada en {OBT_PATH} — {OBT_PATH.stat().st_size/1e6:,.1f} MB, {obt.shape[0]:,} filas × {obt.shape[1]} columnas")

In [ ]:
# Control de completitud temporal: eventos publicados por hora
serie = obt.set_index("FechaPublicacion").resample("1h").size()
fig, ax = plt.subplots(figsize=(11, 3))
serie.plot(ax=ax, lw=1)
ax.set_title("Eventos publicados por hora (huecos = intervalos de 15' faltantes)")
ax.set_ylabel("eventos"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## 2. Limpieza

### 2.1 Nulos y ceros

Primero se perfila **toda la OBT**: porcentaje de nulos y de ceros por columna. Un cero no siempre es un faltante: hay que interpretarlo según el codebook.

In [ ]:
num_cols = obt.select_dtypes("number").columns
perfil = pd.DataFrame({
    "dtype":   obt.dtypes.astype(str),
    "% nulos": obt.isna().mean() * 100,
    "% ceros": (obt[num_cols] == 0).mean().reindex(obt.columns) * 100,
}).sort_values("% nulos", ascending=False)
perfil.round(1)

In [ ]:
null_island = ((obt["ActionGeo_Lat"] == 0) & (obt["ActionGeo_Long"] == 0)).mean() * 100
print(f"ActionGeo en (0,0) 'Null Island': {null_island:.3f} %")
print(f"GoldsteinScale == 0: {(obt['GoldsteinScale'] == 0).mean()*100:.2f} %  (valor válido: evento neutro)")
print(f"AvgTone == 0:        {(obt['AvgTone'] == 0).mean()*100:.2f} %  (valor válido: tono neutro)")
print(f"DiasRetraso < 0:     {(obt['DiasRetraso'] < 0).mean()*100:.3f} %  (inconsistencia temporal)")

top = perfil["% nulos"].head(25)[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top.index, top.values, color="#c0504d")
ax.set_xlabel("% nulos"); ax.set_title("Columnas con más nulos en la OBT")
plt.tight_layout(); plt.show()

**Selección de variables analíticas.** Para la detección de atípicos se trabaja con las variables que describen *qué* pasó, *cuánto* se cubrió, *con qué tono*, *dónde* y *quiénes*:

* **Numéricas:** `GoldsteinScale`, `AvgTone`, `NumMentions`, `NumSources`, `NumArticles`, `DiasRetraso`, `IsRootEvent`, `ActionGeo_Lat`, `ActionGeo_Long`.
* **Categóricas:** `EventRootCode`, `QuadClass`, `Actor1CountryCode`, `Actor2CountryCode`, `ActionGeo_CountryCode`, `Actor1Type1Code`, `ActionGeo_Type`.

Las columnas con nulos masivos (`*KnownGroupCode`, `*EthnicCode`, `*Religion*`, `*Type2/3Code`, geografía de actores) quedan **descartadas** del análisis: superan ampliamente el 50 % de faltantes y su imputación inventaría información. Siguen disponibles en la OBT para interpretar los atípicos.

**Criterios de tratamiento**

| Variable | Situación | Criterio |
|---|---|---|
| `ActionGeo_Lat/Long` | nulos y (0,0) "Null Island" | (0,0) → nulo; imputación por **mediana del país** (`ActionGeo_CountryCode`); si no hay país, se **descarta** la fila (sin ubicación el evento no es localizable) |
| `GoldsteinScale` | nulos esporádicos | imputación por la **moda del `EventCode`**: en CAMEO la escala de Goldstein es función del código de evento |
| `AvgTone` | nulos esporádicos | imputación por **mediana** del `EventRootCode` |
| `NumMentions/Sources/Articles` | ceros o nulos | inconsistentes (todo evento tiene al menos 1 artículo) → **descarte** |
| `DiasRetraso` | nulo o negativo | fecha inválida → **descarte** |
| `GoldsteinScale = 0`, `AvgTone = 0` | ceros | **se conservan**: son valores neutros válidos, no faltantes |
| Categóricas | nulos (p.ej. actor sin país) | categoría explícita **`UNKNOWN`**: el faltante es informativo (MNAR — p.ej. actores no estatales) |

In [ ]:
NUM = ["GoldsteinScale", "AvgTone", "NumMentions", "NumSources", "NumArticles",
       "DiasRetraso", "IsRootEvent", "ActionGeo_Lat", "ActionGeo_Long"]
CAT = ["EventRootCode", "QuadClass", "Actor1CountryCode", "Actor2CountryCode",
       "ActionGeo_CountryCode", "Actor1Type1Code", "ActionGeo_Type"]

df = obt[["GLOBALEVENTID", "EventCode"] + NUM + CAT].copy().set_index("GLOBALEVENTID")
n0 = len(df)
bitacora = []

# (0,0) -> nulo
mask = (df["ActionGeo_Lat"] == 0) & (df["ActionGeo_Long"] == 0)
df.loc[mask, ["ActionGeo_Lat", "ActionGeo_Long"]] = np.nan
bitacora.append(("Null Island -> NaN", int(mask.sum())))

# coordenadas: mediana por país
for c in ["ActionGeo_Lat", "ActionGeo_Long"]:
    falt = df[c].isna().sum()
    df[c] = df[c].fillna(df.groupby("ActionGeo_CountryCode")[c].transform("median"))
    bitacora.append((f"{c} imputada por mediana del país", int(falt - df[c].isna().sum())))
antes = len(df); df = df.dropna(subset=["ActionGeo_Lat", "ActionGeo_Long"])
bitacora.append(("Descartadas sin ubicación", antes - len(df)))

# Goldstein: moda por EventCode
falt = df["GoldsteinScale"].isna().sum()
moda = df.groupby("EventCode")["GoldsteinScale"].agg(lambda s: s.mode().iat[0] if s.notna().any() else np.nan)
df["GoldsteinScale"] = df["GoldsteinScale"].fillna(df["EventCode"].map(moda))
bitacora.append(("GoldsteinScale imputada por moda del EventCode", int(falt - df["GoldsteinScale"].isna().sum())))

# AvgTone: mediana por EventRootCode
falt = df["AvgTone"].isna().sum()
df["AvgTone"] = df["AvgTone"].fillna(df.groupby("EventRootCode")["AvgTone"].transform("median"))
bitacora.append(("AvgTone imputado por mediana del EventRootCode", int(falt - df["AvgTone"].isna().sum())))

# conteos inconsistentes y fechas inválidas
antes = len(df)
cnt = ["NumMentions", "NumSources", "NumArticles"]
df = df[(df[cnt] > 0).all(axis=1) & df["DiasRetraso"].ge(0)]
df = df.dropna(subset=NUM)
bitacora.append(("Descartadas por conteos en 0 / fecha inválida / NaN residual", antes - len(df)))

# categóricas: nulo -> UNKNOWN
df["QuadClass"] = df["QuadClass"].map(QUAD)
df["ActionGeo_Type"] = df["ActionGeo_Type"].map(GEOT)
for c in CAT:
    n = df[c].isna().sum()
    df[c] = df[c].fillna("UNKNOWN").astype(str)
    bitacora.append((f"{c}: nulo -> 'UNKNOWN'", int(n)))

df = df.drop(columns="EventCode")
display(pd.DataFrame(bitacora, columns=["Acción", "Filas afectadas"]))
print(f"Filas: {n0:,} -> {len(df):,}  (descartadas {100*(1-len(df)/n0):.2f} %)")
assert df.isna().sum().sum() == 0

**Transformaciones previas al ACP.** El ACP es sensible a la asimetría y a la escala: las variables de conteo de cobertura mediática tienen colas muy pesadas (pocos eventos acaparan miles de menciones), por lo que se aplica `log1p`. Las coordenadas se pasan a la **esfera unitaria** (x, y, z) para que dos puntos cercanos a ±180° de longitud no queden artificialmente alejados.

In [ ]:
LOG = ["NumMentions", "NumSources", "NumArticles", "DiasRetraso"]
asim_antes = df[LOG].skew()
for c in LOG:
    df[c] = np.log1p(df[c])
asim = pd.DataFrame({"asimetría original": asim_antes, "asimetría log1p": df[LOG].skew()})
display(asim.round(2))

lat, lon = np.radians(df.pop("ActionGeo_Lat")), np.radians(df.pop("ActionGeo_Long"))
df["Geo_x"], df["Geo_y"], df["Geo_z"] = np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)
NUM_F = ["GoldsteinScale", "AvgTone"] + LOG + ["IsRootEvent", "Geo_x", "Geo_y", "Geo_z"]

df[NUM_F].describe().T.round(3)

### 2.2 Reducción de cardinalidad

Las categorías con frecuencia relativa **menor al 5 %** de la muestra se agrupan en `OTHER`. Esto evita cientos de columnas indicadoras casi vacías (p.ej. ~250 códigos de país) que dominarían la varianza del ACP y generarían falsos atípicos por pura rareza de etiqueta.

In [ ]:
def agrupar_marginales(s: pd.Series, umbral: float) -> pd.Series:
    freq = s.value_counts(normalize=True)
    conservar = freq[freq >= umbral].index
    return s.where(s.isin(conservar), "OTHER")

card = []
for c in CAT:
    antes = df[c].nunique()
    df[c] = agrupar_marginales(df[c], UMBRAL_RARO)
    card.append((c, antes, df[c].nunique(), (df[c] == "OTHER").mean() * 100))
display(pd.DataFrame(card, columns=["variable", "cardinalidad original", "cardinalidad reducida", "% en OTHER"])
          .round(1).set_index("variable"))

for c in CAT:
    print(f"\n{c}:", df[c].value_counts(normalize=True).mul(100).round(1).to_dict())

### 2.3 Variables indicadoras (dummies)

Cada categórica se transforma en variables 0/1. Se elimina una **categoría de referencia** por variable (`OTHER` si existe; si no, la más frecuente): con todas las dummies, sus columnas suman 1 y generan colinealidad perfecta (autovalores nulos en el ACP).

In [ ]:
dummies = pd.get_dummies(df[CAT], prefix=CAT, prefix_sep="=", dtype=np.uint8)
referencias = []
for c in CAT:
    ref = f"{c}=OTHER" if f"{c}=OTHER" in dummies else f"{c}={df[c].mode().iat[0]}"
    referencias.append(ref)
dummies = dummies.drop(columns=referencias)
print("Categorías de referencia eliminadas:", referencias)

X = pd.concat([df[NUM_F].astype(np.float32), dummies.astype(np.float32)], axis=1)
print(f"Matriz de características: {X.shape[0]:,} eventos × {X.shape[1]} variables")
X.head()

### 2.4 Reducción de la dimensionalidad: ACP

Se estandarizan todas las características (media 0, desvío 1) para que ninguna domine por su escala, y se ajusta el ACP con todas las componentes. Se comparan cuatro criterios para la cantidad mínima de componentes:

* **Varianza acumulada** ≥ 80 % (criterio adoptado) y ≥ 90 %.
* **Kaiser:** autovalores > 1 (la componente explica más que una variable original estandarizada).
* **Codo** del *scree plot* (máxima distancia a la recta entre el primer y el último autovalor).

In [ ]:
scaler = StandardScaler()
Xs = scaler.fit_transform(X.values).astype(np.float32)

try:
    pca = PCA(svd_solver="covariance_eigh", random_state=SEED).fit(Xs)   # eficiente con n >> p
except (ValueError, TypeError):
    pca = PCA(svd_solver="full", random_state=SEED).fit(Xs)

autoval = pca.explained_variance_
evr = pca.explained_variance_ratio_
cum = evr.cumsum()

def codo(v):
    x = np.arange(len(v)); x1, y1, x2, y2 = 0, v[0], len(v) - 1, v[-1]
    d = np.abs((y2 - y1) * x - (x2 - x1) * v + x2 * y1 - y2 * x1) / np.hypot(y2 - y1, x2 - x1)
    return int(d.argmax()) + 1

criterios = {
    "Varianza acumulada ≥ 80 %": int(np.searchsorted(cum, 0.80) + 1),
    "Varianza acumulada ≥ 90 %": int(np.searchsorted(cum, 0.90) + 1),
    "Kaiser (λ > 1)":            int((autoval > 1).sum()),
    "Codo (scree plot)":         codo(autoval),
}
K = int(np.searchsorted(cum, UMBRAL_VAR) + 1)
display(pd.Series(criterios, name="componentes").to_frame())
print(f"\n==> K = {K} componentes (de {X.shape[1]} variables) explican {cum[K-1]*100:.1f} % de la varianza")

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4))
idx = np.arange(1, len(autoval) + 1)
ax1.bar(idx, evr * 100, color="#4f81bd", alpha=0.8, label="varianza explicada")
ax1.set_xlabel("componente principal"); ax1.set_ylabel("% varianza explicada")
ax2 = ax1.twinx()
ax2.plot(idx, cum * 100, "o-", color="#c0504d", ms=3, label="acumulada")
ax2.axhline(UMBRAL_VAR * 100, ls="--", color="gray")
ax2.axvline(K, ls=":", color="black")
ax2.set_ylabel("% acumulado"); ax2.set_ylim(0, 102); ax2.grid(False)
ax1.set_title(f"Scree plot y varianza acumulada — K = {K}")
fig.legend(loc="center right", bbox_to_anchor=(0.88, 0.5))
plt.tight_layout(); plt.show()

In [ ]:
# Cargas (loadings) de las primeras componentes: correlación variable-componente
nc = min(K, 8)
loadings = pd.DataFrame(pca.components_[:nc].T * np.sqrt(autoval[:nc]),
                        index=X.columns, columns=[f"CP{i+1}" for i in range(nc)])
fig, ax = plt.subplots(figsize=(1.0 * nc + 3, 0.28 * len(X.columns) + 1))
im = ax.imshow(loadings.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(nc)); ax.set_xticklabels(loadings.columns)
ax.set_yticks(range(len(X.columns))); ax.set_yticklabels(X.columns, fontsize=7)
ax.grid(False); plt.colorbar(im, ax=ax, label="carga")
ax.set_title("Cargas de las variables en las primeras componentes")
plt.tight_layout(); plt.show()

for c in loadings.columns[:4]:
    top = loadings[c].abs().sort_values(ascending=False).head(4).index
    print(f"{c} ({evr[int(c[2:])-1]*100:.1f} %):", ", ".join(f"{v} ({loadings.at[v, c]:+.2f})" for v in top))

**Interpretación.** Las cargas permiten nombrar las componentes: típicamente una componente de **cobertura mediática** (NumMentions/NumSources/NumArticles), una de **conflicto vs. cooperación** (GoldsteinScale, QuadClass, EventRootCode, AvgTone), y componentes **geográficas** (Geo_x/y/z y país del ActionGeo). Al trabajar con una matriz mixta (continuas + dummies), la varianza se reparte entre muchas direcciones, por lo que K suele ser una fracción moderada de las variables originales; aun así el ACP elimina las direcciones redundantes y de ruido.

### 2.5 Detección de eventos atípicos multivariantes

Sobre el modelo ACP con K componentes se usan los dos estadísticos clásicos de control multivariante, más un método no paramétrico:

1. **T² de Hotelling** — distancia de Mahalanobis *dentro* del subespacio de K componentes: $T^2_i = \sum_{k=1}^{K} z_{ik}^2/\lambda_k$. Límite: $\chi^2_{0.999}(K)$. Detecta eventos con combinaciones **extremas pero coherentes** con la estructura (p.ej. cobertura masiva).
2. **SPE / estadístico Q** — error cuadrático de reconstrucción *fuera* del subespacio: $Q_i = \lVert x_i - \hat x_i \rVert^2$. Límite por la aproximación de Box $g\,\chi^2_{\alpha}(h)$ con $g = \mathrm{var}(Q)/(2\bar Q)$, $h = 2\bar Q^2/\mathrm{var}(Q)$. Detecta eventos que **rompen la correlación** habitual (p.ej. tono muy positivo en un evento de conflicto material).
3. **Isolation Forest** sobre los *scores* — no supone normalidad; se marca el 0,1 % más aislado.

Como los datos de noticias tienen colas pesadas y los límites paramétricos suponen normalidad, se exige **consenso: un evento es atípico si al menos 2 de los 3 métodos lo marcan**. El ranking final es el promedio de los percentiles de los tres puntajes.

In [ ]:
Z = pca.transform(Xs)[:, :K].astype(np.float64)
lam = autoval[:K]

T2 = (Z**2 / lam).sum(axis=1)
lim_T2 = chi2.ppf(ALPHA, K)

Xc = Xs.astype(np.float64) - pca.mean_
SPE = (Xc**2).sum(axis=1) - (Z**2).sum(axis=1)          # proyección ortonormal: ||x||² = ||z||² + ||e||²
SPE = np.clip(SPE, 0, None)
m, v = SPE.mean(), SPE.var()
g, h = v / (2 * m), 2 * m**2 / v
lim_SPE = g * chi2.ppf(ALPHA, h)

rng = np.random.default_rng(SEED)
muestra = rng.choice(len(Z), size=min(len(Z), 300_000), replace=False)
iso = IsolationForest(n_estimators=300, max_samples=512, random_state=SEED, n_jobs=-1).fit(Z[muestra])
IF = -iso.score_samples(Z)                                 # mayor = más anómalo
lim_IF = np.quantile(IF, 1 - FRAC_IF)

res = pd.DataFrame({"T2": T2, "SPE": SPE, "IF_score": IF}, index=df.index)
res["flag_T2"], res["flag_SPE"], res["flag_IF"] = T2 > lim_T2, SPE > lim_SPE, IF > lim_IF
res["votos"] = res[["flag_T2", "flag_SPE", "flag_IF"]].sum(axis=1)
res["score_atipico"] = res[["T2", "SPE", "IF_score"]].rank(pct=True).mean(axis=1)
res["atipico"] = res["votos"] >= 2

print(f"Límite T²  (χ² {ALPHA}, K={K}): {lim_T2:,.2f} -> {res.flag_T2.mean()*100:.2f} % marcados")
print(f"Límite SPE (Box):              {lim_SPE:,.2f} -> {res.flag_SPE.mean()*100:.2f} % marcados")
print(f"Límite Isolation Forest:       {lim_IF:,.4f} -> {res.flag_IF.mean()*100:.2f} % marcados")
display(res["votos"].value_counts().sort_index().rename("eventos").to_frame())
print(f"\n==> Atípicos de consenso (≥2 votos): {res.atipico.sum():,} ({res.atipico.mean()*100:.3f} %)")

In [ ]:
# Gráfico de control T² vs SPE (muestra + todos los atípicos)
sel = np.union1d(rng.choice(len(res), size=min(len(res), 60_000), replace=False),
                 np.flatnonzero(res["atipico"].values))
r = res.iloc[sel]
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(r.T2[~r.atipico], r.SPE[~r.atipico], s=3, alpha=0.25, color="#7f7f7f", label="normal (muestra)")
ax.scatter(r.T2[r.atipico], r.SPE[r.atipico], s=10, color="#c0504d", label="atípico (consenso)")
ax.axvline(lim_T2, ls="--", color="black", lw=1); ax.axhline(lim_SPE, ls="--", color="black", lw=1)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("T² de Hotelling"); ax.set_ylabel("SPE (Q)")
ax.set_title("Diagrama de control multivariante"); ax.legend()
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(Z[sel, 0], Z[sel, 1], s=3, alpha=0.25, c=np.where(res["atipico"].values[sel], "#c0504d", "#7f7f7f"))
ax.set_xlabel(f"CP1 ({evr[0]*100:.1f} %)"); ax.set_ylabel(f"CP2 ({evr[1]*100:.1f} %)")
ax.set_title("Eventos en el plano CP1–CP2 (rojo = atípico)")
plt.tight_layout(); plt.show()

#### Explicación de cada atípico y exportación

Para que la tabla sea accionable por un periodista, cada atípico lleva:
* las 3 variables con mayor **|z-score|** (qué tiene de extremo), y
* las 3 variables con mayor **residuo** de reconstrucción (qué correlación rompe),

junto con **todos los atributos de la OBT** (actores, lugar, descripción CAMEO, URL de la fuente).

In [ ]:
ids = res.index[res["atipico"]]
pos = res.index.get_indexer(ids)
cols = np.array(X.columns)

zs = Xs[pos]
resid = Xc[pos] - Z[pos] @ pca.components_[:K]

def top3(mat):
    idx = np.argsort(-np.abs(mat), axis=1)[:, :3]
    return [", ".join(f"{cols[j]} ({fila[j]:+.1f})" for j in fila_idx) for fila, fila_idx in zip(mat, idx)]

explic = pd.DataFrame({"Top_zscore": top3(zs), "Top_residuo": top3(resid)}, index=ids)

COLS_OBT = ["GLOBALEVENTID", "FechaEvento", "FechaPublicacion", "DiasRetraso",
            "Actor1Name", "Actor1CountryCode", "Actor1CountryDesc", "Actor2Name", "Actor2CountryCode", "Actor2CountryDesc",
            "EventCode", "EventDesc", "EventRootCode", "EventRootDesc", "QuadClassDesc",
            "GoldsteinScale", "AvgTone", "NumMentions", "NumSources", "NumArticles", "IsRootEvent",
            "ActionGeo_FullName", "ActionGeo_CountryCode", "ActionGeo_CountryDesc", "ActionGeo_Lat", "ActionGeo_Long",
            "Dominio", "SOURCEURL"]

atipicos = (obt[COLS_OBT].set_index("GLOBALEVENTID").loc[ids]
            .join(res.loc[ids, ["T2", "SPE", "IF_score", "flag_T2", "flag_SPE", "flag_IF", "votos", "score_atipico"]])
            .join(explic)
            .sort_values(["votos", "score_atipico"], ascending=False)
            .reset_index())
atipicos.insert(1, "ranking", np.arange(1, len(atipicos) + 1))

atipicos.to_parquet(ATI_PATH, engine="pyarrow", compression="snappy", index=False)
atipicos.to_csv(ATI_PATH.with_suffix(".csv"), index=False)
print(f"Tabla de atípicos exportada: {ATI_PATH} y {ATI_PATH.with_suffix('.csv')} — {len(atipicos):,} eventos")
atipicos.head(25)

#### Perfil de los atípicos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

base = obt.set_index("GLOBALEVENTID").loc[df.index]
lift = (atipicos["EventRootDesc"].value_counts(normalize=True) /
        base["EventRootDesc"].value_counts(normalize=True)).dropna().sort_values().tail(12)
axes[0].barh(lift.index, lift.values, color="#4f81bd"); axes[0].axvline(1, color="black", lw=1)
axes[0].set_title("Sobre-representación (lift) por tipo de evento"); axes[0].tick_params(axis="y", labelsize=8)

pais = atipicos["ActionGeo_CountryDesc"].fillna("Sin dato").value_counts().head(12)[::-1]
axes[1].barh(pais.index, pais.values, color="#c0504d"); axes[1].set_title("Atípicos por país del evento")
axes[1].tick_params(axis="y", labelsize=8)

fondo = base.sample(min(len(base), 50_000), random_state=SEED)
axes[2].scatter(fondo["ActionGeo_Long"], fondo["ActionGeo_Lat"], s=1, alpha=0.15, color="#7f7f7f")
axes[2].scatter(atipicos["ActionGeo_Long"], atipicos["ActionGeo_Lat"],
                s=8 + 30 * atipicos["score_atipico"], color="#c0504d", alpha=0.7)
axes[2].set_xlim(-180, 180); axes[2].set_ylim(-90, 90)
axes[2].set_title("Ubicación de los atípicos"); axes[2].set_xlabel("longitud"); axes[2].set_ylabel("latitud")
plt.tight_layout(); plt.show()

comp = pd.DataFrame({
    "todos (mediana)":    base[["GoldsteinScale", "AvgTone", "NumMentions", "NumSources", "NumArticles", "DiasRetraso"]].median(),
    "atípicos (mediana)": atipicos[["GoldsteinScale", "AvgTone", "NumMentions", "NumSources", "NumArticles", "DiasRetraso"]].median(),
})
display(comp.round(2))

In [ ]:
# Tipología: qué método(s) marcó cada atípico
tipo = (atipicos[["flag_T2", "flag_SPE", "flag_IF"]]
        .apply(lambda f: " + ".join(n for n, x in zip(["T²", "SPE", "IF"], f) if x), axis=1)
        .value_counts().rename("eventos"))
display(tipo.to_frame())

# Los 10 eventos más raros, listos para investigar
(atipicos.head(10)[["ranking", "FechaEvento", "Actor1Name", "Actor2Name", "EventDesc", "QuadClassDesc",
                    "GoldsteinScale", "AvgTone", "NumArticles", "ActionGeo_FullName", "Top_zscore", "SOURCEURL"]])

## 3. Conclusiones

* **Ingesta.** Se construyó una OBT de los últimos 7 días a partir de las tablas *Event* de 15 minutos, verificando integridad (MD5) y completitud temporal, y desnormalizando las dimensiones CAMEO/FIPS. El formato Parquet columnar reduce fuertemente el volumen respecto de los TSV y deja la tabla lista para consultas analíticas.
* **Calidad de datos.** Los faltantes de GDELT no son aleatorios: se concentran en atributos de actores no estatales y en la geografía de actores. Se distinguieron ceros estructurales (Goldstein y tono neutros) de ceros que codifican faltantes (Null Island), y se imputó usando conocimiento del dominio (Goldstein es función del código CAMEO).
* **Cardinalidad y dummies.** El umbral del 5 % redujo cientos de categorías a unas pocas por variable, evitando que la rareza de una etiqueta sea, por sí sola, causa de "atipicidad".
* **ACP.** El número mínimo de componentes se fijó con el criterio de 80 % de varianza acumulada y se contrastó con Kaiser y el codo (ver tabla de criterios). Las primeras componentes se interpretan como cobertura mediática, polaridad conflicto/cooperación y ubicación.
* **Atípicos.** La combinación de T² (extremos dentro del modelo), SPE (rupturas de la estructura de correlación) e Isolation Forest (no paramétrico), con regla de consenso, produce una lista corta y rankeada de eventos. Cada uno incluye la explicación de las variables que lo vuelven raro y la URL de la fuente, lo que permite al equipo periodístico priorizar la verificación.
* **Limitaciones.** Los límites χ² suponen normalidad multivariante, que los datos no cumplen (colas pesadas); por eso se usó consenso y ranking en lugar de un umbral único. Un atípico estadístico puede ser un error de codificación automática de GDELT (actor o lugar mal geocodificado) y no un hecho noticioso: la verificación humana sobre `SOURCEURL` es parte del proceso.

In [ ]:
# Resumen numérico de la corrida
print(f"Ventana: {obt['FechaPublicacion'].min()} -> {obt['FechaPublicacion'].max()}")
print(f"Archivos de 15': {len(archivos) - len(fallidos)} / {len(archivos)}")
print(f"OBT: {len(obt):,} eventos × {obt.shape[1]} columnas")
print(f"Matriz analítica: {X.shape[0]:,} × {X.shape[1]}  |  K = {K} componentes ({cum[K-1]*100:.1f} % var.)")
print(f"Atípicos de consenso: {len(atipicos):,} ({len(atipicos)/len(X)*100:.3f} %)")

In [ ]:
# Descarga opcional de los productos (sólo en Colab)
try:
    from google.colab import files
    files.download(str(ATI_PATH.with_suffix(".csv")))
except ImportError:
    print("No se está ejecutando en Colab; los archivos quedan en", DATA_DIR.resolve())

## Referencias

* GDELT Project. *GDELT 2.0 Event Database* — lista de archivos: http://data.gdeltproject.org/gdeltv2/masterfilelist.txt
* GDELT Project. *GDELT Event Codebook V2.0*: https://data.gdeltproject.org/documentation/GDELT-Event_Codebook-V2.0.pdf
* Schrodt, P. A. *CAMEO Conflict and Mediation Event Observations Codebook v1.1b3*: https://gdeltproject.org/data/documentation/CAMEO.Manual.1.1b3.pdf
* Jackson, J. E. & Mudholkar, G. S. (1979). Control procedures for residuals associated with principal component analysis. *Technometrics*, 21(3).
* Liu, F. T., Ting, K. M. & Zhou, Z.-H. (2008). Isolation Forest. *ICDM*.